In [1]:
import sys, os
sys.path.insert(0, os.path.join('..'))   # project root on path

import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import wrds
import polars as pl
import pyarrow
import config

from sklearn.preprocessing import StandardScaler

from src.data_loading import load_crsp, load_futures, load_crsp_polars, wrds_fetch,load_cz_monthly
from src.preprocessing import clean_crsp, clean_futures
from src.feature_engineering import add_target, add_volatility_momentum, crosssectional_rank, get_feature_cols 
from src.utils import generate_date_split, split_batch

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
features = pd.read_parquet(config.FEATURES_PATH_CLEAN, )
print(f'Number of unique PERMNO in dataset : {features['PERMNO'].nunique()}')
print(f'Initial RAM space taken is : {features.memory_usage(index=True).sum() / 1024**3:.2f} Gb')

Number of unique PERMNO in dataset : 6647
Initial RAM space taken is : 2.91 Gb


In [3]:
num_cols = features.select_dtypes('number').columns
features[num_cols] = features[num_cols].astype('float32') # Halving of RAM space taken by features dataframe
print(f'RAM space taken is now : {features.memory_usage(index=True).sum() / 1024**3:.2f} Gb')

RAM space taken is now : 1.63 Gb


In [4]:
cz_daily = load_cz_monthly()
print(f'Initial RAM space taken is : {cz_daily.memory_usage(index=True).sum() / 1024**3:.2f} Gb')
num_cols = cz_daily.select_dtypes('number').columns
cz_daily[num_cols] = cz_daily[num_cols].astype('float32') # Halving of RAM space taken by features dataframe
print(f'RAM space taken is now : {cz_daily.memory_usage(index=True).sum() / 1024**3:.2f} Gb')

Initial Size of the dataset: (1140, 206)
Total Date range : 1926-01-30 00:00:00 -> 2020-12-31 00:00:00
Dataset shape after dropping column with less than 90.0% completion: (1140, 57)
Total column dropped so far : 149
Dataset shape after dropping highly correlated (corr_coef > 0.95) columns: (1140, 55)
Total column dropped so far : 151
Initial RAM space taken is : 0.01 Gb
RAM space taken is now : 0.01 Gb


In [5]:
cz_indexed = cz_daily.set_index('date').sort_index()
cz_indexed.index = pd.to_datetime(cz_indexed.index).astype('datetime64[ms]') # Prepare to merge

In [6]:
print("features date dtype:", features.index.get_level_values('date').dtype 
      if isinstance(features.index, pd.MultiIndex) 
      else features['date'].dtype)
print("cz_daily date dtype:", cz_daily.index.dtype)
print("cz_daily index sample:", cz_daily.index[:3])

features date dtype: datetime64[ms]
cz_daily date dtype: int64
cz_daily index sample: Index([2191, 2192, 2193], dtype='int64')


In [7]:
split_batch = split_batch(df = features, df_to_join=[cz_indexed], batch_number=2)

Total unique dates: 6089
  train: 2000-10-16 → 2017-09-25  (4262 dates, 70.0%)
  val  : 2017-09-26 → 2021-05-12  (913 dates, 15.0%)
  test : 2021-05-13 → 2024-12-30  (914 dates, 15.0%)
Total unique dates: 6089
  train: 2000-10-16 → 2017-09-25  (4262 dates, 70.0%)
  val  : 2017-09-26 → 2021-05-12  (913 dates, 15.0%)
  test : 2021-05-13 → 2024-12-30  (914 dates, 15.0%)


In [13]:
split_batch['batch_1']['validation'].isna().sum()

ret                  0
mkt_ret              0
reversal_1d          0
mom_scaled_5d        0
mom_5d               0
                  ... 
VolSD             9145
VolumeTrend       9145
zerotrade         9145
zerotradeAlt1     9145
zerotradeAlt12    9145
Length: 73, dtype: int64